In [0]:
dbutils.widgets.text('sourceFilePath','/mnt/raw/iothub/FACT_DeviceTwinEvents/')
sourceFilePath = dbutils.widgets.get('sourceFilePath')

dbutils.widgets.text('ConfigId','16')
ConfigId = dbutils.widgets.get('ConfigId')

dbutils.widgets.text('SourceTypeId','6')
sourceTypeId = dbutils.widgets.get('SourceTypeId')

dbutils.widgets.text('startingVersion','0')
startingVersion = dbutils.widgets.get('startingVersion')
print("startingVersion:"+str(startingVersion))

job_id=dbutils.widgets.text("job_id","-1")
job_id=dbutils.widgets.get("job_id")

run_id=dbutils.widgets.text("run_id","-1")
run_id=dbutils.widgets.get("run_id")

dbutils.widgets.text('EmailNotificationID','3')
EmailNotificationID = dbutils.widgets.get('EmailNotificationID')

dbutils.widgets.text('Env','Dev')
Env = dbutils.widgets.get('Env')

In [0]:
import traceback

pipelinename = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
pipelinename = pipelinename.rsplit('/', 1)[-1]
display(pipelinename)

In [0]:
%run ../../Configurations/Init_Scripts

In [0]:
%run
/Configurations/EmailNotificationConfiguration

In [0]:
checkPointLocation = "/_checkpoints/"
destinationFilePath = '/mnt/silver/FACTDeviceTwin'


# Define File Schema

In [0]:
deviceTwinSchema = StructType([
        StructField('etag', StringType(), True), 
        StructField('device_id', StringType(), True), 
        StructField('status', StringType(), True),
        StructField('status_update_time', StringType(), True), 
        StructField('connection_state', StringType(), True),
        StructField('last_activity_time', StringType(), True), 
        StructField('cloud_to_device_message_count', LongType(), True), 
        StructField('authentication_type', StringType(), True), 
        StructField('version', LongType(), True), 

        StructField('capabilities',StructType([StructField('iot_edge', StringType(), True)]),True),        
        StructField('properties',StructType(
        [
            StructField('desired',StructType(
            [
                StructField('account', StringType(), True), 
                StructField('key', StringType(), True), 
                StructField('formatVersion', StringType(), True), 
                StructField('upload', StringType(), True), 
                StructField('twinPollingRate', LongType(), True), 
                StructField('systemLock',StringType(), True), 
                StructField('packages', StringType(), True),               
                StructField('deviceEvents',StringType(), True)
            ]),True),
            StructField('reported',StructType(
            [
                StructField('assignedIoTHub', StringType(), True),
                StructField('packages', StringType(), True),
                StructField('deviceEvents',StringType(), True),                
                StructField('formatVersion', StringType(), True), 
                StructField('totalChannels', LongType(), True), 
                StructField('pagerEnabled', BooleanType(), True), 
                StructField('systemType', LongType(), True), 
                StructField('externalSerialNumber', StringType(), True),
                StructField('internalSerialNumber', StringType(), True), 
                StructField('aliasName', StringType(), True), 
                StructField('language', StringType(), True), 
                StructField('phoneHomeZCode', StructType([StructField('major', LongType(), True), 
                                                          StructField('minor', LongType(), True)]), True), 
                StructField('systemLockStatus', StringType(), True), 
                StructField('releaseVersion', StringType(), True),
                StructField('FluidFilter', StringType(), True),
                StructField('AirFilter', StringType(), True),               
                StructField('HandPiece', StringType(), True),
                StructField('ElectrodeConnector', StringType(), True),

                StructField('Versions',
                StructType(
                    [
                        StructField('App1', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('App2', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('AuxiliaryApp', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('CCB', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('Configurator', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('Cooler', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('ENCApp', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('ExtInterfaceApp', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('Installer', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('LogExtractor', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('LogExtractorUI', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('LogSVC', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('Maestro', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('MedApp', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('MedAppUI', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('OEMSecurityService', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('OS', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('PIBFPGA', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('PIBTec1FPGA', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('PIBTec2FPGA', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('Pib', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('PsbHubService', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), 
                        True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('SecurityService', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('Vac1', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                        
                        StructField('Vac2', StructType([StructField('Valid', BooleanType(), True), StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True)]), True), 
                                    
                        StructField('DevicePowerManager', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                        
                        StructField('FMS', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                        
                        StructField('PPS_MCU', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                        
                        StructField('PPS_DSP', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True)
                    ]), True),
                StructField('softwareVersions',
                            StructType(
                                [
                                    StructField('App1', StructType([StructField('boot', LongType(), True), 
                                                                    StructField('fw', StructType([StructField('md', LongType(), True), 
                                                                                                    StructField('mj', LongType(), True), 
                                                                                                    StructField('mn', LongType(), True)]), True),
                                                                        StructField('nv', LongType(), True), 
                                                                        StructField('valid', BooleanType(), True)]), True), 
                                    StructField('App2', StructType([StructField('boot', LongType(), True), 
                                                                    StructField('fw', StructType([StructField('md', LongType(), True), 
                                                                                                    StructField('mj', LongType(), True), 
                                                                                                    StructField('mn', LongType(), True)]), True),
                                                                        StructField('nv', LongType(), True), 
                                                                        StructField('valid', BooleanType(), True)]), True), 
                                    StructField('AuxiliaryApp', StructType([StructField('boot', LongType(), True), 
                                                                            StructField('fw', StructType([StructField('md', LongType(), True),
                                                                                                            StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), 
                                                                            StructField('nv', LongType(), True), 
                                                                            StructField('valid', BooleanType(), True)]), True), 
                                    StructField('CCB', StructType([StructField('boot', LongType(), True), 
                                                                    StructField('fw', StructType([StructField('md', LongType(), True), 
                                                                                                    StructField('mj', LongType(), True), 
                                                                                                    StructField('mn', LongType(), True)]), True), 
                                                                    StructField('nv', LongType(), True), 
                                                                    StructField('valid', BooleanType(), True)]), True), 
                                    StructField('Configurator', StructType([StructField('boot', LongType(), True), 
                                                                            StructField('fw', StructType([StructField('md', LongType(), True), 
                                                                                                            StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), 
                                                                            StructField('nv', LongType(), True), 
                                                                            StructField('valid', BooleanType(), True)]), True), 
                                    StructField('Cooler', StructType([StructField('boot', LongType(), True), 
                                                                        StructField('fw', StructType([StructField('md', LongType(), True), 
                                                                                                    StructField('mj', LongType(), True), 
                                                                                                    StructField('mn', LongType(), True)]), True), 
                                                                        StructField('nv', LongType(), True), 
                                                                        StructField('valid', BooleanType(), True)]), True), 
                                    StructField('ENCApp', StructType([StructField('boot', LongType(), True), 
                                                                        StructField('fw', StructType([StructField('md', LongType(), True), 
                                                                                                    StructField('mj', LongType(), True), 
                                                                                                    StructField('mn', LongType(), True)]), True), 
                                                                        StructField('nv', LongType(), True), 
                                                                        StructField('valid', BooleanType(), True)]), True), 
                                    StructField('ExtInterfaceApp', StructType([StructField('boot', LongType(), True), 
                                                                        StructField('fw', StructType([StructField('md', LongType(), True), 
                                                                                                    StructField('mj', LongType(), True), 
                                                                                                    StructField('mn', LongType(), True)]), True), 
                                                                        StructField('nv', LongType(), True), 
                                                                        StructField('valid', BooleanType(), True)]), True), 
                                    StructField('Installer', StructType([StructField('boot', LongType(), True), 
                                                                        StructField('fw', StructType([StructField('md', LongType(), True), 
                                                                                                    StructField('mj', LongType(), True), 
                                                                                                    StructField('mn', LongType(), True)]), True), 
                                                                        StructField('nv', LongType(), True), 
                                                                        StructField('valid', BooleanType(), True)]), True), 
                                    StructField('LogExtractor', StructType([StructField('boot', LongType(), True), 
                                                                            StructField('fw', StructType([StructField('md', LongType(), True), 
                                                                                                        StructField('mj', LongType(), True), 
                                                                                                        StructField('mn', LongType(), True)]), True), 
                                                                            StructField('nv', LongType(), True), 
                                                                            StructField('valid', BooleanType(), True)]), True), 
                                    StructField('LogExtractorUI', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 

                                    StructField('LogSVC', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('Maestro', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('MedApp', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', 
                                    LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('MedAppUI', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('OEMSecurityService', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('OS', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('PIBFPGA', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('PIBTec1FPGA', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('PIBTec2FPGA', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('Pib', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('PsbHubService', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('SecurityService', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('Vac1', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('Vac2', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('DevicePowerManager', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('FMS', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('PPS_MCU', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True), 
                                    
                                    StructField('PPS_DSP', StructType([StructField('boot', LongType(), True), StructField('fw', StructType([StructField('md', LongType(), True), StructField('mj', LongType(), True), StructField('mn', LongType(), True)]), True), StructField('nv', LongType(), True), StructField('valid', BooleanType(), True)]), True)

                                ]), True),
            ]),True),

        ]),True),
    ])

In [0]:
df_Source = (spark.readStream.option("cloudFiles.maxFilesPerTrigger",5000) 
                              .option("cloudFiles.maxBytesPerTrigger",'10g')
                                .option("startingVersion", startingVersion)
                                .option("skipChangeCommits", "true")
                               .format("delta").load(sourceFilePath))


In [0]:
def Rename_DeviceTwinLogs(df_fullLoad):
    df_fullLoad_WithCols = (df_fullLoad
                .withColumn('twinData',from_json(col("twinData"),deviceTwinSchema))
                .withColumn('Etag',col("twinData.etag"))
                .withColumn('DeviceId',trim(upper(col("device_id"))))
                .withColumn('StatusFlag',col("twinData.status"))
                .withColumn('StatusUpdatedTime',col("twinData.status_update_time").cast('TIMESTAMP'))
                .withColumn('ConnectionState',col("twinData.connection_state"))
                .withColumn('LastActivityTime',col("twinData.last_activity_time").cast('TIMESTAMP'))
                .withColumn('CloudToDeviceMessageCount',col("twinData.cloud_to_device_message_count").cast('LONG'))
                .withColumn('AuthenticationType',col("twinData.authentication_type"))
                .withColumn('VersionNbr',col("twinData.version"))
                .withColumn('Properties_Desired_Account',col("twinData.properties.desired.account"))
                .withColumn('Properties_Desired_Key',col("twinData.properties.desired.key"))
                .withColumn('Properties_Desired_FormatVersion',col("twinData.properties.desired.formatVersion"))
                .withColumn("Properties_Desired_Packages", col("twinData.properties.desired.packages").cast('string'))
                .withColumn("Properties_Desired_Upload", col("twinData.properties.desired.upload").cast('string'))
                .withColumn('Properties_Desired_TwinPollingRate',col("twinData.properties.desired.twinPollingRate"))
                .withColumn('Properties_Desired_SystemLock',col("twinData.properties.desired.systemLock").cast('string'))
                .withColumn('Properties_Desired_DeviceEvents',col("twinData.properties.desired.deviceEvents").cast('string'))
                .withColumn('Properties_Reported_AssignedIoTHub',col("twinData.properties.reported.assignedIoTHub"))
                .withColumn('Properties_Reported_FormatVersion',col("twinData.properties.reported.formatVersion"))
                .withColumn('Properties_Reported_TotalChannels',col("twinData.properties.reported.totalChannels"))
                .withColumn('Properties_Reported_PagerEnabled',col("twinData.properties.reported.pagerEnabled").cast('string'))
                .withColumn('Properties_Reported_ProductType',col("productType"))
                .withColumn('Properties_Reported_SystemType',col("twinData.properties.reported.systemType"))
                .withColumn('ExternalSerialNbr',trim(upper(col("twinData.properties.reported.externalSerialNumber"))))
                .withColumn('InternalSerialNbr',trim(upper(col("twinData.properties.reported.internalSerialNumber"))))
                .withColumn('Properties_Reported_AliasName',col("twinData.properties.reported.aliasName"))
                .withColumn('Properties_Reported_Language',col("twinData.properties.reported.language"))
                .withColumn('Properties_Reported_PhoneHomeZCode',col("twinData.properties.reported.phoneHomeZCode"))
                .withColumn('PhoneHomeZMajorVerCd',col("twinData.properties.reported.phoneHomeZCode.major"))
                .withColumn('PhoneHomeZMinorVerCd',col("twinData.properties.reported.phoneHomeZCode.minor"))

                .withColumn("Properties_Reported_Packages", col("twinData.properties.reported.packages").cast('string'))
                .withColumn('Properties_Reported_SystemLockStatus',col("twinData.properties.reported.systemLockStatus"))
                .withColumn('Properties_Reported_ReleaseVersion',col("twinData.properties.reported.releaseVersion"))
                .withColumn('Properties_Reported_DeviceEvents',col("twinData.properties.reported.deviceEvents").cast('string'))
                .withColumn('Properties_Reported_FluidFilter',col("twinData.properties.reported.FluidFilter").cast('string'))
                .withColumn('Properties_Reported_AirFilter',col("twinData.properties.reported.AirFilter").cast('string'))
                .withColumn('Properties_Reported_HandPiece',col("twinData.properties.reported.HandPiece").cast('string'))
                .withColumn('Properties_Reported_ElectrodeConnector',col("twinData.properties.reported.ElectrodeConnector").cast('string'))
                .withColumn('Capabilities_IOTEdge',col("twinData.capabilities.iot_edge").cast('string'))
                .withColumn('Properties_Reported_Softwareversions',col('twinData.properties.reported.softwareVersions').cast('string'))
                .withColumn('Properties_Reported_Versions',col('twinData.properties.reported.Versions').cast('string'))                
                .withColumn('App1Version', when(col("twinData.properties.reported.Softwareversions.App1").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.App1.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.App1.fw.md")
                                                    ,col("twinData.properties.reported.Versions.App1.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.App1.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.App1.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.App1.fw.mn"))))
                .withColumn('App2Version', when(col("twinData.properties.reported.Softwareversions.App2").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.App2.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.App2.fw.md")
                                                    ,col("twinData.properties.reported.Versions.App2.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.App2.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.App2.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.App2.fw.mn"))))
                .withColumn('AuxiliaryAppVersion', when(col("twinData.properties.reported.Softwareversions.AuxiliaryApp").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.AuxiliaryApp.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.AuxiliaryApp.fw.md")
                                                    ,col("twinData.properties.reported.Versions.AuxiliaryApp.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.AuxiliaryApp.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.AuxiliaryApp.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.AuxiliaryApp.fw.mn"))))
                .withColumn('CCBVersion', when(col("twinData.properties.reported.Softwareversions.CCB").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.CCB.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.CCB.fw.md")
                                                    ,col("twinData.properties.reported.Versions.CCB.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.CCB.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.CCB.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.CCB.fw.mn"))))
                .withColumn('ConfiguratorVersion', when(col("twinData.properties.reported.Softwareversions.Configurator").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.Configurator.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.Configurator.fw.md")
                                                    ,col("twinData.properties.reported.Versions.Configurator.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.Configurator.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.Configurator.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.Configurator.fw.mn"))))
            .withColumn('CoolerVersion', when(col("twinData.properties.reported.Softwareversions.Cooler").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.Cooler.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.Cooler.fw.md")
                                                    ,col("twinData.properties.reported.Versions.Cooler.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.Cooler.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.Cooler.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.Cooler.fw.mn"))))
            .withColumn('ENCAppVersion', when(col("twinData.properties.reported.Softwareversions.ENCApp").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.ENCApp.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.ENCApp.fw.md")
                                                    ,col("twinData.properties.reported.Versions.ENCApp.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.ENCApp.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.ENCApp.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.ENCApp.fw.mn"))))
            .withColumn('InstallerVersion', when(col("twinData.properties.reported.Softwareversions.Installer").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.Installer.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.Installer.fw.md")
                                                    ,col("twinData.properties.reported.Versions.Installer.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.Installer.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.Installer.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.Installer.fw.mn"))))
            .withColumn('LogExtractorVersion', when(col("twinData.properties.reported.Softwareversions.LogExtractor").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.LogExtractor.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.LogExtractor.fw.md")
                                                    ,col("twinData.properties.reported.Versions.LogExtractor.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.LogExtractor.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.LogExtractor.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.LogExtractor.fw.mn"))))
            .withColumn('LogExtractorUIVersion', when(col("twinData.properties.reported.Softwareversions.LogExtractorUI").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.LogExtractorUI.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.LogExtractorUI.fw.md")
                                                    ,col("twinData.properties.reported.Versions.LogExtractorUI.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.LogExtractorUI.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.LogExtractorUI.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.LogExtractorUI.fw.mn"))))
            .withColumn('LogSVCVersion', when(col("twinData.properties.reported.Softwareversions.LogSVC").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.LogSVC.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.LogSVC.fw.md")
                                                    ,col("twinData.properties.reported.Versions.LogSVC.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.LogSVC.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.LogSVC.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.LogSVC.fw.mn"))))
            .withColumn('MaestroVersion', when(col("twinData.properties.reported.Softwareversions.Maestro").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.Maestro.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.Maestro.fw.md")
                                                    ,col("twinData.properties.reported.Versions.Maestro.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.Maestro.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.Maestro.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.Maestro.fw.mn"))))           
            .withColumn('MedAppVersion', when(col("twinData.properties.reported.Softwareversions.MedApp").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.MedApp.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.MedApp.fw.md")
                                                    ,col("twinData.properties.reported.Versions.MedApp.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.MedApp.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.MedApp.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.MedApp.fw.mn"))))
            .withColumn('MedAppUIVersion', when(col("twinData.properties.reported.Softwareversions.MedAppUI").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.MedAppUI.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.MedAppUI.fw.md")
                                                    ,col("twinData.properties.reported.Versions.MedAppUI.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.MedAppUI.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.MedAppUI.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.MedAppUI.fw.mn"))))
            .withColumn('OSVersion', when(col("twinData.properties.reported.Softwareversions.OS").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.OS.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.OS.fw.md")
                                                    ,col("twinData.properties.reported.Versions.OS.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.OS.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.OS.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.OS.fw.mn"))))
            .withColumn('PIBFPGAVersion', when(col("twinData.properties.reported.Softwareversions.PIBFPGA").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.PIBFPGA.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.PIBFPGA.fw.md")
                                                    ,col("twinData.properties.reported.Versions.PIBFPGA.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.PIBFPGA.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.PIBFPGA.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.PIBFPGA.fw.mn"))))
            .withColumn('PIBTec1FPGAVersion', when(col("twinData.properties.reported.Softwareversions.PIBTec1FPGA").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.PIBTec1FPGA.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.PIBTec1FPGA.fw.md")
                                                    ,col("twinData.properties.reported.Versions.PIBTec1FPGA.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.PIBTec1FPGA.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.PIBTec1FPGA.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.PIBTec1FPGA.fw.mn"))))
            .withColumn('PIBTec2FPGAVersion', when(col("twinData.properties.reported.Softwareversions.PIBTec2FPGA").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.PIBTec2FPGA.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.PIBTec2FPGA.fw.md")
                                                    ,col("twinData.properties.reported.Versions.PIBTec2FPGA.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.PIBTec2FPGA.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.PIBTec2FPGA.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.PIBTec2FPGA.fw.mn"))))
            .withColumn('PibVersion', when(col("twinData.properties.reported.Softwareversions.Pib").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.Pib.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.Pib.fw.md")
                                                    ,col("twinData.properties.reported.Versions.Pib.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.Pib.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.Pib.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.Pib.fw.mn"))))
            .withColumn('PsbHubServiceVersion', when(col("twinData.properties.reported.Softwareversions.PsbHubService").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.PsbHubService.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.PsbHubService.fw.md")
                                                    ,col("twinData.properties.reported.Versions.PsbHubService.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.PsbHubService.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.PsbHubService.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.PsbHubService.fw.mn")))) 
            .withColumn('SecurityServiceVersion', when(col("twinData.properties.reported.Softwareversions.SecurityService").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.SecurityService.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.SecurityService.fw.md")
                                                    ,col("twinData.properties.reported.Versions.SecurityService.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.SecurityService.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.SecurityService.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.SecurityService.fw.mn")))) 
            .withColumn('Vac1Version', when(col("twinData.properties.reported.Softwareversions.Vac1").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.Vac1.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.Vac1.fw.md")
                                                    ,col("twinData.properties.reported.Versions.Vac1.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.Vac1.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.Vac1.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.Vac1.fw.mn")))) 
            .withColumn('Vac2Version', when(col("twinData.properties.reported.Softwareversions.Vac2").isNull(),
                                        concat_ws('.',col("twinData.properties.reported.Versions.Vac2.fw.mj")
                                                    ,col("twinData.properties.reported.Versions.Vac2.fw.md")
                                                    ,col("twinData.properties.reported.Versions.Vac2.fw.mn")))
                            .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.Vac2.fw.mj")
                                                    ,col("twinData.properties.reported.Softwareversions.Vac2.fw.md")
                                                    ,col("twinData.properties.reported.Softwareversions.Vac2.fw.mn")))) 
            .withColumn('DevicePowerManagerVersion', 
                    when(col("twinData.properties.reported.Softwareversions.DevicePowerManager").isNull(),
                                concat_ws('.',col("twinData.properties.reported.Versions.DevicePowerManager.fw.mj")
                                            ,col("twinData.properties.reported.Versions.DevicePowerManager.fw.md")
                                            ,col("twinData.properties.reported.Versions.DevicePowerManager.fw.mn")))
                    .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.DevicePowerManager.fw.mj")
                                            ,col("twinData.properties.reported.Softwareversions.DevicePowerManager.fw.md")
                                            ,col("twinData.properties.reported.Softwareversions.DevicePowerManager.fw.mn"))))

            .withColumn('FMSVersion', 
                    when(col("twinData.properties.reported.Softwareversions.FMS").isNull(),
                                concat_ws('.',col("twinData.properties.reported.Versions.FMS.fw.mj")
                                            ,col("twinData.properties.reported.Versions.FMS.fw.md")
                                            ,col("twinData.properties.reported.Versions.FMS.fw.mn")))
                    .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.FMS.fw.mj")
                                            ,col("twinData.properties.reported.Softwareversions.FMS.fw.md")
                                            ,col("twinData.properties.reported.Softwareversions.FMS.fw.mn"))))
            .withColumn('PPS_MCUVersion', 
                    when(col("twinData.properties.reported.Softwareversions.PPS_MCU").isNull(),
                                concat_ws('.',col("twinData.properties.reported.Versions.PPS_MCU.fw.mj")
                                            ,col("twinData.properties.reported.Versions.PPS_MCU.fw.md")
                                            ,col("twinData.properties.reported.Versions.PPS_MCU.fw.mn")))
                    .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.PPS_MCU.fw.mj")
                                            ,col("twinData.properties.reported.Softwareversions.PPS_MCU.fw.md")
                                            ,col("twinData.properties.reported.Softwareversions.PPS_MCU.fw.mn"))))
            .withColumn('PPS_DSPVersion', 
                    when(col("twinData.properties.reported.Softwareversions.PPS_DSP").isNull(),
                                concat_ws('.',col("twinData.properties.reported.Versions.PPS_DSP.fw.mj")
                                            ,col("twinData.properties.reported.Versions.PPS_DSP.fw.md")
                                            ,col("twinData.properties.reported.Versions.PPS_DSP.fw.mn")))
                    .otherwise(concat_ws('.',col("twinData.properties.reported.Softwareversions.PPS_DSP.fw.mj")
                                            ,col("twinData.properties.reported.Softwareversions.PPS_DSP.fw.md")
                                            ,col("twinData.properties.reported.Softwareversions.PPS_DSP.fw.mn"))))               
            .withColumn("DtTmstmp",current_timestamp())
            .withColumn('DeviceTwinDt',current_date())
            .withColumn("CreatedBy",lit("ADB_DeviceTwin"))
            .withColumn("CreatedDt",current_timestamp())
            .withColumn("UpdatedBy",lit("ADB_DeviceTwin"))
            .withColumn("UpdatedDt",current_timestamp())
            .drop("twinData").dropDuplicates()
            
            )
    return df_fullLoad_WithCols

In [0]:
def ProcessFullLoad(DF_DeviceTwin_RenameCols,destinationFilePath):
    DedupSeq = (Window.partitionBy("DeviceId").orderBy(desc(col('CreatedDt'))))
    DF_DeviceTwin_FullLoad = (DF_DeviceTwin_RenameCols.filter("operationType = 'FullLoad'"))
    DF_DeviceTwin_FullLoad_Latest = (DF_DeviceTwin_FullLoad.withColumn("rowNumDedup", row_number().over(DedupSeq))
                                                            .filter('rowNumDedup=1')
                                                            .drop('rowNumDedup')
                                                            .withColumn('CreatedDt',lit(current_timestamp()))
                                                            .withColumn('UpdatedDt',lit(current_timestamp()))
                                    )
    
    DF_DeviceTwin_Destination = DeltaTable.forPath(spark, destinationFilePath)

    (DF_DeviceTwin_Destination.alias("tgt")
                .merge(DF_DeviceTwin_FullLoad_Latest.alias("src"),
                        "tgt.DeviceId = src.DeviceId"
                        )
                .whenNotMatchedInsert(values ={
                    # "tgt.FileNameUUID" : "src.FileNameUUID",
                    # "tgt.SourceFilePath" : "src.SourceFilePath",
                    # "tgt.SourceFileName" : "src.SourceFileName",
                    # "tgt.file_modification_time" : "src.file_modification_time",
                    # "tgt.SourceFileSize" : "src.SourceFileSize",
                    "tgt.Etag" : "src.Etag",
                    "tgt.DeviceID" : "src.DeviceID",
                    "tgt.StatusFlag" : "src.StatusFlag",
                    "tgt.StatusUpdatedTime" : "src.StatusUpdatedTime",
                    "tgt.ConnectionState" : "src.ConnectionState",
                    "tgt.LastActivityTime" : "src.LastActivityTime",
                    "tgt.CloudToDeviceMessageCount" : "src.CloudToDeviceMessageCount",
                    "tgt.AuthenticationType" : "src.AuthenticationType",
                    "tgt.VersionNbr" : "src.VersionNbr",
                    "tgt.Properties_Desired_Account" : "src.Properties_Desired_Account",
                    "tgt.Properties_Desired_Key" : "src.Properties_Desired_Key",
                    "tgt.Properties_Desired_FormatVersion" : "src.Properties_Desired_FormatVersion",
                    "tgt.Properties_Desired_Packages" : "src.Properties_Desired_Packages",
                    "tgt.Properties_Desired_Upload" : "src.Properties_Desired_Upload",
                    "tgt.Properties_Desired_TwinPollingRate" : "src.Properties_Desired_TwinPollingRate",
                    "tgt.Properties_Reported_AssignedIoTHub" : "src.Properties_Reported_AssignedIoTHub",
                    "tgt.Properties_Reported_FormatVersion" : "src.Properties_Reported_FormatVersion",
                    "tgt.Properties_Reported_PagerEnabled" : "src.Properties_Reported_PagerEnabled",
                    "tgt.Properties_Desired_SystemLock" : "src.Properties_Desired_SystemLock",
                    "tgt.Properties_Desired_DeviceEvents" : "src.Properties_Desired_DeviceEvents",
                    "tgt.Properties_Reported_TotalChannels" : "src.Properties_Reported_TotalChannels",
                    "tgt.Properties_Reported_SystemLockStatus" : "src.Properties_Reported_SystemLockStatus",
                    "tgt.Properties_Reported_DeviceEvents" : "src.Properties_Reported_DeviceEvents",
                    "tgt.Properties_Reported_ProductType" : "src.Properties_Reported_ProductType",
                    "tgt.Properties_Reported_SystemType" : "src.Properties_Reported_SystemType",
                    "tgt.ExternalSerialNbr" : "src.ExternalSerialNbr",
                    "tgt.InternalSerialNbr" : "src.InternalSerialNbr",
                    "tgt.Properties_Reported_AliasName" : "src.Properties_Reported_AliasName",
                    "tgt.Properties_Reported_Language" : "src.Properties_Reported_Language",
                    "tgt.Properties_Reported_Packages" : "src.Properties_Reported_Packages",
                    "tgt.Properties_Reported_ReleaseVersion" : "src.Properties_Reported_ReleaseVersion",

                    "tgt.Properties_Reported_FluidFilter" : "src.Properties_Reported_FluidFilter",
                    "tgt.Properties_Reported_AirFilter" : "src.Properties_Reported_AirFilter",
                    "tgt.Properties_Reported_HandPiece" : "src.Properties_Reported_HandPiece",
                    "tgt.Properties_Reported_ElectrodeConnector" : "src.Properties_Reported_ElectrodeConnector",

                    "tgt.Capabilities_IOTEdge" : "src.Capabilities_IOTEdge",
                    "tgt.PhoneHomeZMajorVerCd" : "src.PhoneHomeZMajorVerCd",
                    "tgt.PhoneHomeZMinorVerCd" : "src.PhoneHomeZMinorVerCd",
                    "tgt.DtTmstmp" : "src.DtTmstmp",
                    "tgt.DeviceTwinDt" : "src.DeviceTwinDt",
                    "tgt.App1Version" : "src.App1Version",
                    "tgt.App2Version" : "src.App2Version",
                    "tgt.AuxiliaryAppVersion" : "src.AuxiliaryAppVersion",
                    "tgt.CCBVersion" : "src.CCBVersion",
                    "tgt.ConfiguratorVersion" : "src.ConfiguratorVersion",
                    "tgt.CoolerVersion" : "src.CoolerVersion",
                    "tgt.ENCAppVersion" : "src.ENCAppVersion",
                    "tgt.InstallerVersion" : "src.InstallerVersion",
                    "tgt.LogExtractorVersion" : "src.LogExtractorVersion",
                    "tgt.LogExtractorUIVersion" : "src.LogExtractorUIVersion",
                    "tgt.LogSVCVersion" : "src.LogSVCVersion",
                    "tgt.MaestroVersion" : "src.MaestroVersion",
                    "tgt.MedAppVersion" : "src.MedAppVersion",
                    "tgt.MedAppUIVersion" : "src.MedAppUIVersion",
                    "tgt.OSVersion" : "src.OSVersion",
                    "tgt.PIBFPGAVersion" : "src.PIBFPGAVersion",
                    "tgt.PIBTec1FPGAVersion" : "src.PIBTec1FPGAVersion",
                    "tgt.PIBTec2FPGAVersion" : "src.PIBTec2FPGAVersion",
                    "tgt.PibVersion" : "src.PibVersion",
                    "tgt.PsbHubServiceVersion" : "src.PsbHubServiceVersion",
                    "tgt.SecurityServiceVersion" : "src.SecurityServiceVersion",
                    "tgt.Vac1Version" : "src.Vac1Version",
                    "tgt.Vac2Version" : "src.Vac2Version",

                    "tgt.DevicePowerManagerVersion" : "src.DevicePowerManagerVersion",
                    "tgt.FMSVersion" : "src.FMSVersion",
                    "tgt.PPS_MCUVersion" : "src.PPS_MCUVersion",
                    "tgt.PPS_DSPVersion" : "src.PPS_DSPVersion",

                    "tgt.CreatedBy" : "src.operationType",
                    "tgt.CreatedDt" : "src.CreatedDt",
                    "tgt.UpdatedBy" : "src.operationType",
                    "tgt.UpdatedDt" : "src.UpdatedDt"
                    })
                .whenMatchedUpdate(set = {"tgt.Etag" : "src.Etag",
                                        "tgt.DeviceID" : "src.DeviceID",
                                        "tgt.StatusFlag" : "src.StatusFlag",
                                        "tgt.StatusUpdatedTime" : "src.StatusUpdatedTime",
                                        "tgt.ConnectionState" : "src.ConnectionState",
                                        "tgt.LastActivityTime" : "src.LastActivityTime",
                                        "tgt.CloudToDeviceMessageCount" : "src.CloudToDeviceMessageCount",
                                        "tgt.AuthenticationType" : "src.AuthenticationType",
                                        "tgt.VersionNbr" : "src.VersionNbr",
                                        "tgt.Properties_Desired_Account" : "src.Properties_Desired_Account",
                                        "tgt.Properties_Desired_Key" : "src.Properties_Desired_Key",
                                        "tgt.Properties_Desired_FormatVersion" : "src.Properties_Desired_FormatVersion",
                                        "tgt.Properties_Desired_Packages" : "src.Properties_Desired_Packages",
                                        "tgt.Properties_Desired_Upload" : "src.Properties_Desired_Upload",
                                        "tgt.Properties_Desired_TwinPollingRate" : "src.Properties_Desired_TwinPollingRate",
                                        "tgt.Properties_Reported_AssignedIoTHub" : "src.Properties_Reported_AssignedIoTHub",
                                        "tgt.Properties_Reported_FormatVersion" : "src.Properties_Reported_FormatVersion",
                                        "tgt.Properties_Reported_PagerEnabled" : "src.Properties_Reported_PagerEnabled",
                                        "tgt.Properties_Desired_SystemLock" : "src.Properties_Desired_SystemLock",
                                        "tgt.Properties_Desired_DeviceEvents" : "src.Properties_Desired_DeviceEvents",
                                        "tgt.Properties_Reported_TotalChannels" : "src.Properties_Reported_TotalChannels",
                                        "tgt.Properties_Reported_SystemLockStatus" : "src.Properties_Reported_SystemLockStatus",
                                        "tgt.Properties_Reported_DeviceEvents" : "src.Properties_Reported_DeviceEvents",
                                        "tgt.Properties_Reported_ProductType" : "src.Properties_Reported_ProductType",
                                        "tgt.Properties_Reported_SystemType" : "src.Properties_Reported_SystemType",
                                        "tgt.ExternalSerialNbr" : "src.ExternalSerialNbr",
                                        "tgt.InternalSerialNbr" : "src.InternalSerialNbr",
                                        "tgt.Properties_Reported_AliasName" : "src.Properties_Reported_AliasName",
                                        "tgt.Properties_Reported_Language" : "src.Properties_Reported_Language",
                                        "tgt.Properties_Reported_Packages" : "src.Properties_Reported_Packages",
                                        "tgt.Properties_Reported_ReleaseVersion" : "src.Properties_Reported_ReleaseVersion",

                                        "tgt.Properties_Reported_FluidFilter" : "src.Properties_Reported_FluidFilter",
                                        "tgt.Properties_Reported_AirFilter" : "src.Properties_Reported_AirFilter",
                                        "tgt.Properties_Reported_HandPiece" : "src.Properties_Reported_HandPiece",
                                        "tgt.Properties_Reported_ElectrodeConnector" : "src.Properties_Reported_ElectrodeConnector",

                                        "tgt.Capabilities_IOTEdge" : "src.Capabilities_IOTEdge",
                                        "tgt.PhoneHomeZMajorVerCd" : "src.PhoneHomeZMajorVerCd",
                                        "tgt.PhoneHomeZMinorVerCd" : "src.PhoneHomeZMinorVerCd",
                                        "tgt.DtTmstmp" : "src.DtTmstmp",
                                        "tgt.DeviceTwinDt" : "src.DeviceTwinDt",
                                        "tgt.App1Version" : "src.App1Version",
                                        "tgt.App2Version" : "src.App2Version",
                                        "tgt.AuxiliaryAppVersion" : "src.AuxiliaryAppVersion",
                                        "tgt.CCBVersion" : "src.CCBVersion",
                                        "tgt.ConfiguratorVersion" : "src.ConfiguratorVersion",
                                        "tgt.CoolerVersion" : "src.CoolerVersion",
                                        "tgt.ENCAppVersion" : "src.ENCAppVersion",
                                        "tgt.InstallerVersion" : "src.InstallerVersion",
                                        "tgt.LogExtractorVersion" : "src.LogExtractorVersion",
                                        "tgt.LogExtractorUIVersion" : "src.LogExtractorUIVersion",
                                        "tgt.LogSVCVersion" : "src.LogSVCVersion",
                                        "tgt.MaestroVersion" : "src.MaestroVersion",
                                        "tgt.MedAppVersion" : "src.MedAppVersion",
                                        "tgt.MedAppUIVersion" : "src.MedAppUIVersion",
                                        "tgt.OSVersion" : "src.OSVersion",
                                        "tgt.PIBFPGAVersion" : "src.PIBFPGAVersion",
                                        "tgt.PIBTec1FPGAVersion" : "src.PIBTec1FPGAVersion",
                                        "tgt.PIBTec2FPGAVersion" : "src.PIBTec2FPGAVersion",
                                        "tgt.PibVersion" : "src.PibVersion",
                                        "tgt.PsbHubServiceVersion" : "src.PsbHubServiceVersion",
                                        "tgt.SecurityServiceVersion" : "src.SecurityServiceVersion",
                                        "tgt.Vac1Version" : "src.Vac1Version",
                                        "tgt.Vac2Version" : "src.Vac2Version",

                                        "tgt.DevicePowerManagerVersion" : "src.DevicePowerManagerVersion",
                                        "tgt.FMSVersion" : "src.FMSVersion",
                                        "tgt.PPS_MCUVersion" : "src.PPS_MCUVersion",
                                        "tgt.PPS_DSPVersion" : "src.PPS_DSPVersion",

                                        "tgt.CreatedBy" : "src.operationType",
                                        "tgt.CreatedDt" : "src.CreatedDt",
                                        "tgt.UpdatedBy" : "src.operationType",
                                        "tgt.UpdatedDt" : "src.UpdatedDt"})
                .execute()
                )


In [0]:
def ProcessIncremental(DF_DeviceTwin_RenameCols,destinationFilePath):


    DF_DeviceTwin_Incremental_Agg = (DF_DeviceTwin_RenameCols
                                        .filter('''
                                                operationType = 'updateTwin'
                                                and (StatusFlag is not null
                                                        or Properties_Reported_Softwareversions is not null 
                                                        or Properties_Reported_Versions is not null
                                                        or ConnectionState is not null
                                                        or LastActivityTime is not null
                                                        or Properties_Desired_Packages is not null
                                                        or Properties_Reported_Packages is not null
                                                        or Properties_Reported_FluidFilter is not null
                                                        or Properties_Reported_AirFilter is not null
                                                        or Properties_Reported_HandPiece is not null
                                                        or Properties_Reported_ElectrodeConnector is not null)
                                             ''')
                                        .groupBy("DeviceId","operationType")
                                        .agg(
                                                        max("VersionNbr").alias("VersionNbr"),
                                                        max("StatusFlag").alias("StatusFlag"),
                                                        max("ConnectionState").alias("ConnectionState"),
                                                        max("LastActivityTime").alias("LastActivityTime"),
                                                        max("Properties_Desired_Packages").alias("Properties_Desired_Packages"),
                                                        max("Properties_Reported_Packages").alias("Properties_Reported_Packages"),
                                                        max("Properties_Reported_FluidFilter").alias("Properties_Reported_FluidFilter"),
                                                        max("Properties_Reported_AirFilter").alias("Properties_Reported_AirFilter"),
                                                        max("Properties_Reported_HandPiece").alias("Properties_Reported_HandPiece"),
                                                        max("Properties_Reported_ElectrodeConnector").alias("Properties_Reported_ElectrodeConnector"),
                                                        max("Properties_Reported_Softwareversions").alias("Properties_Reported_Softwareversions"),
                                                        max("Properties_Reported_Versions").alias("Properties_Reported_Versions"),
                                                        max("App1Version").alias("App1Version"),
                                                        max("App2Version").alias("App2Version"),
                                                        max("AuxiliaryAppVersion").alias("AuxiliaryAppVersion"),
                                                        max("CCBVersion").alias("CCBVersion"),
                                                        max("ConfiguratorVersion").alias("ConfiguratorVersion"),
                                                        max("CoolerVersion").alias("CoolerVersion"),
                                                        max("ENCAppVersion").alias("ENCAppVersion"),
                                                        max("InstallerVersion").alias("InstallerVersion"),
                                                        max("LogExtractorVersion").alias("LogExtractorVersion"),
                                                        max("LogExtractorUIVersion").alias("LogExtractorUIVersion"),
                                                        max("LogSVCVersion").alias("LogSVCVersion"),
                                                        max("MaestroVersion").alias("MaestroVersion"),
                                                        max("MedAppVersion").alias("MedAppVersion"),
                                                        max("MedAppUIVersion").alias("MedAppUIVersion"),
                                                        max("OSVersion").alias("OSVersion"),
                                                        max("PIBFPGAVersion").alias("PIBFPGAVersion"),
                                                        max("PIBTec1FPGAVersion").alias("PIBTec1FPGAVersion"),
                                                        max("PIBTec2FPGAVersion").alias("PIBTec2FPGAVersion"),
                                                        max("PibVersion").alias("PibVersion"),
                                                        max("PsbHubServiceVersion").alias("PsbHubServiceVersion"),
                                                        max("SecurityServiceVersion").alias("SecurityServiceVersion"),
                                                        max("Vac1Version").alias("Vac1Version"),
                                                        max("Vac2Version").alias("Vac2Version"),
                                                        max("DevicePowerManagerVersion").alias("DevicePowerManagerVersion"),
                                                        max("FMSVersion").alias("FMSVersion"),
                                                        max("PPS_MCUVersion").alias("PPS_MCUVersion"),
                                                        max("PPS_DSPVersion").alias("PPS_DSPVersion")
                                             )
                                        .withColumn('UpdatedDt',lit(current_timestamp()))
                                        )


    
    DF_DeviceTwin_Destination = DeltaTable.forPath(spark, destinationFilePath)
    (DF_DeviceTwin_Destination.alias("tgt")
    .merge(DF_DeviceTwin_Incremental_Agg.alias("src"),
            "tgt.DeviceId = src.DeviceId"
            )
    .whenMatchedUpdate( condition = "src.Properties_Reported_Softwareversions is not null or src.Properties_Reported_Versions is not null",
                    set = {
                            "tgt.App1Version" : "IFNULL(src.App1Version,tgt.App1Version)",
                            "tgt.App2Version" : "IFNULL(src.App2Version,tgt.App2Version)",
                            "tgt.AuxiliaryAppVersion" : "IFNULL(src.AuxiliaryAppVersion,tgt.AuxiliaryAppVersion)",
                            "tgt.CCBVersion" : "IFNULL(src.CCBVersion,tgt.CCBVersion)",
                            "tgt.ConfiguratorVersion" : "IFNULL(src.ConfiguratorVersion,tgt.ConfiguratorVersion)",
                            "tgt.CoolerVersion" : "IFNULL(src.CoolerVersion,tgt.CoolerVersion)",
                            "tgt.ENCAppVersion" : "IFNULL(src.ENCAppVersion,tgt.ENCAppVersion)",
                            "tgt.InstallerVersion" : "IFNULL(src.InstallerVersion,tgt.InstallerVersion)",
                            "tgt.LogExtractorVersion" : "IFNULL(src.LogExtractorVersion,tgt.LogExtractorVersion)",
                            "tgt.LogExtractorUIVersion" : "IFNULL(src.LogExtractorUIVersion,tgt.LogExtractorUIVersion)",
                            "tgt.LogSVCVersion" : "IFNULL(src.LogSVCVersion,tgt.LogSVCVersion)",
                            "tgt.MaestroVersion" : "IFNULL(src.MaestroVersion,tgt.MaestroVersion)",
                            "tgt.MedAppVersion" : "IFNULL(src.MedAppVersion,tgt.MedAppVersion)",
                            "tgt.MedAppUIVersion" : "IFNULL(src.MedAppUIVersion,tgt.MedAppUIVersion)",
                            "tgt.OSVersion" : "IFNULL(src.OSVersion,tgt.OSVersion)",
                            "tgt.PIBFPGAVersion" : "IFNULL(src.PIBFPGAVersion,tgt.PIBFPGAVersion)",
                            "tgt.PIBTec1FPGAVersion" : "IFNULL(src.PIBTec1FPGAVersion,tgt.PIBTec1FPGAVersion)",
                            "tgt.PIBTec2FPGAVersion" : "IFNULL(src.PIBTec2FPGAVersion,tgt.PIBTec2FPGAVersion)",
                            "tgt.PibVersion" : "IFNULL(src.PibVersion,tgt.PibVersion)",
                            "tgt.PsbHubServiceVersion" : "IFNULL(src.PsbHubServiceVersion,tgt.PsbHubServiceVersion)",
                            "tgt.SecurityServiceVersion" : "IFNULL(src.SecurityServiceVersion,tgt.SecurityServiceVersion)",
                            "tgt.Vac1Version" : "IFNULL(src.Vac1Version,tgt.Vac1Version)",
                            "tgt.Vac2Version" : "IFNULL(src.Vac2Version,tgt.Vac2Version)",

                            "tgt.DevicePowerManagerVersion" : "IFNULL(src.DevicePowerManagerVersion,tgt.DevicePowerManagerVersion)",
                            "tgt.FMSVersion" : "IFNULL(src.FMSVersion,tgt.FMSVersion)",
                            "tgt.PPS_MCUVersion" : "IFNULL(src.PPS_MCUVersion,tgt.PPS_MCUVersion)",
                            "tgt.PPS_DSPVersion" : "IFNULL(src.PPS_DSPVersion,tgt.PPS_DSPVersion)",

                            "tgt.UpdatedBy" : "src.operationType",
                            "tgt.UpdatedDt" : "src.UpdatedDt"})                                              
    .execute()
    )

    (DF_DeviceTwin_Destination.alias("tgt")
    .merge(DF_DeviceTwin_Incremental_Agg.alias("src"),
            "tgt.DeviceId = src.DeviceId"
            ) 
    .whenMatchedUpdate( set = {
                            "tgt.StatusFlag" : "IFNULL(src.StatusFlag,tgt.StatusFlag)",
                            "tgt.ConnectionState" : "IFNULL(src.ConnectionState,tgt.ConnectionState)",
                            "tgt.LastActivityTime" : "IFNULL(src.LastActivityTime,tgt.LastActivityTime)",
                            "tgt.VersionNbr" : "IFNULL(src.VersionNbr,tgt.VersionNbr)",
                            "tgt.Properties_Desired_Packages" : "IFNULL(src.Properties_Desired_Packages,tgt.Properties_Desired_Packages)",
                            "tgt.Properties_Reported_Packages" : "IFNULL(src.Properties_Reported_Packages,tgt.Properties_Reported_Packages)",
                            "tgt.Properties_Reported_FluidFilter" : "IFNULL(src.Properties_Reported_FluidFilter,tgt.Properties_Reported_FluidFilter)",
                            "tgt.Properties_Reported_AirFilter" : "IFNULL(src.Properties_Reported_AirFilter,tgt.Properties_Reported_AirFilter)",
                            "tgt.Properties_Reported_HandPiece" : "IFNULL(src.Properties_Reported_HandPiece,tgt.Properties_Reported_HandPiece)",
                            "tgt.Properties_Reported_ElectrodeConnector" : "IFNULL(src.Properties_Reported_ElectrodeConnector,tgt.Properties_Reported_ElectrodeConnector)",
                            "tgt.UpdatedBy" : "src.operationType",
                            "tgt.UpdatedDt" : "src.UpdatedDt"})                                              
    .execute()
    )    



In [0]:
def Process_DeviceTwinLogs(microBatchOutputDF,batchId):
        FileNameUUID = str(uuid.uuid4())
        df_logSource = (microBatchOutputDF
                        .withColumn('SourceFilePath', lit(sourceFilePath))
                        .withColumn('SourceFileName', lit('DeviceTwin'))
                        .withColumn('SourceFileSize', lit(0).cast(LongType()))
                        .withColumn("FileNameDeviceTypeCd",lit('DeviceTwin'))
                        .withColumn("FileNameDeviceSerialNbr",lit('DeviceTwin'))
                        .withColumn("FileNameMessageTypeCd",lit('DeviceTwin'))
                        .withColumn("FileNameDtTmstmp",lit(None))
                        .withColumn("FileNameApplicatorPortCd",lit('DeviceTwin'))
                        .withColumn("FileNameCycleNbr",lit(None))
                        .withColumn("DeviceType",lit('DeviceTwin'))
                        .withColumn("LogType",lit('DeviceTwin'))
                        .withColumn("DeviceId",lit('DeviceTwin'))
                        .withColumn('SourceTypeId',lit(sourceTypeId))
                        .withColumn("HdrAppHeadNbr",lit('DeviceTwin'))
                        .withColumn('InternalSerialNbr',lit(None))
                        .withColumn('ExternalSerialNbr',lit(None))
                        .withColumn('RunID',lit(batchId))
                        .withColumn("LogStartDate",lit(None))
                        .withColumn("LogEndDate",lit(None))
                        .withColumn('ConfigId',lit(ConfigId))
                        .withColumn("LogStartDate",lit(None))
                        .withColumn("LogEndDate",lit(None))
                        .withColumn("FileNameUUID",lit(FileNameUUID))
                        .withColumn("RawFileModificationTime",lit(None))                        
                        
                )
        
        df_logSource_CNT = (df_logSource.groupBy('FileNameDeviceTypeCd','ExternalSerialNbr','InternalSerialNbr','FileNameMessageTypeCd','FileNameDtTmstmp','FileNameApplicatorPortCd',
                                                'FileNameCycleNbr','LogStartDate','LogEndDate','SourceFilePath','SourceFileName','SourceFileSize','SourceTypeId','FileNameUUID',
                                                'DeviceType','LogType','DeviceId','ConfigId',"RawFileModificationTime","RunID")
                        .count()
                        .withColumnRenamed("count","RecdCnt")                            
                        )
        df_logSource_CNT.persist()
        loadlogProcessesDeltaTable(df_logSource_CNT,destinationFilePath,'ADB_LoadDeviceTwin','InProgress','')
        loadAuditTables_Ingestion_Log(df_logSource_CNT,destinationFilePath,'ADB_LoadDeviceTwin','InProgress','')
        try:
                log_df = df_logSource.select(col('ConfigId').alias('ConfigID'), col('SourceTypeId').alias('SourceTypeID'), col('SourceFilePath').alias('Source')) \
                                         .withColumn('Destination', lit(str(destinationFilePath))) \
                                         .withColumn('Run_ID', lit(str(batchId))) \
                                         .withColumn('Job_ID', lit(str(job_id)))
        
                Microbatch_df = df_logSource

                # raise Exception("No Exception: Manual Failure")

                DF_DeviceTwin_RenameCols = Rename_DeviceTwinLogs(microBatchOutputDF)
                ProcessFullLoad(DF_DeviceTwin_RenameCols,destinationFilePath)
                ProcessIncremental(DF_DeviceTwin_RenameCols,destinationFilePath)

                loadAuditTables_Ingestion_Log(df_logSource_CNT,destinationFilePath,'ADB_LoadDeviceTwin','Succeeded','')
                loadlogProcessesDeltaTable(df_logSource_CNT,destinationFilePath,'ADB_LoadDeviceTwin','Succeeded','')
        except Exception as exp:
                ExceptionTraceback = traceback.format_exc()
                ErrorMessage = ExceptionTraceback + str(exp)
                loadAuditTables_Ingestion_Log(df_logSource_CNT,destinationFilePath,'ADB_LoadDeviceTwin','Failed',str(exp))
                loadlogProcessesDeltaTable(df_logSource_CNT,destinationFilePath,'ADB_LoadDeviceTwin','Failed',str(exp))
                logIntoStreamLogTable(log_df,"ADB_LoadDeviceTwin","Failed",Microbatch_df,ErrorMessage)
                streamLogEmailNotification(EmailNotificationID,Microbatch_df, pipelinename, Env)
                print(ExceptionTraceback)
                # raise            


# Master function to call all process logs in order

In [0]:
def upsertToDelta(microBatchOutputDF, batchId):     
    try:
        batchEnd(q,batchId)
        print("Running for BatchID: {0}".format(batchId))
        Process_DeviceTwinLogs(microBatchOutputDF,batchId)
        spark.sql("clear cache")
    except Exception as exp:
        Print(str(exp))
        # raise        

# Streaming job to process log data

In [0]:
q=(df_Source.writeStream
                  .format("delta")
                  .queryName("Transformation_DeviceTwin_Stream")
                  .trigger(processingTime='10 seconds')
                  .foreachBatch(upsertToDelta)
                  .option("checkpointLocation", destinationFilePath+checkPointLocation)
                  .outputMode("update")
                  .start()
                #   .awaitTermination()
)

In [0]:
stop_process = 1
stop_process =  graceStop(q,1)

In [0]:
q.awaitTermination()